# Xgrammar + vLLM Adapter Demo

Demonstrates three features of `VLLMXgrammarAdapter`:
1. **count_branches(prefix)** — KB-constrained prefix query tool
2. **JSON mode** — xgrammar GrammarMatcher for structured output
3. **Fact: triple mode** — KB-constrained fact extraction

All run without a GPU (logic-only tests).

In [ ]:
import math
import torch
import sys
sys.path.insert(0, "..")
from transformers import AutoTokenizer
from refactx.index import DictIndex, EmptyIndexException, TripleNotFoundException
from e2e_vllm_report import VLLMXgrammarAdapter

print("Imports OK")

## 1. Setup: Tokenizer + KB

In [ ]:
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", trust_remote_code=True)
vocab_size = len(tok)
print(f"Vocab size: {vocab_size}")

kb = DictIndex()
triples = [
    "<Paris> <capital of> <France>",
    "<France> <continent> <Europe>",
    "<Mont Blanc> <elevation> <4808 meters>",
    "<Europe> <contains> <France>",
    "<France> <contains> <Paris>",
]
for t in triples + [" " + t for t in triples]:  # both space/non-space forms
    kb.add(tok.encode(t, add_special_tokens=False))

print(f"KB leaves: {kb.count_leaves()}")

## 2. count_branches(prefix) — core logic

The helper `_count_at_prefix` traverses the KB trie and returns the number of leaves under any prefix.

In [ ]:
def count_at_prefix(index, prefix):
    """Number of KB leaves under a given prefix."""
    level = index.tree
    cursor = 0
    lc = 0
    while cursor < len(prefix) and lc < len(level[1]):
        if isinstance(level[1][lc], dict):
            if prefix[cursor] in level[1][lc]:
                level = level[1][lc][prefix[cursor]]
                lc = 0
            else:
                return 0
        else:
            if prefix[cursor] != level[1][lc]:
                return 0
            lc += 1
        cursor += 1
    return level[0]


tests = [
    ([], "empty (all KB leaves)"),
    (tok.encode("<France>", add_special_tokens=False), "<France>"),
    (tok.encode("<Europe>", add_special_tokens=False), "<Europe>"),
    (tok.encode("<Paris>", add_special_tokens=False), "<Paris>"),
    (tok.encode("<France> <continent>", add_special_tokens=False), "<France> <continent>"),
]

print("count_branches results:")
for prefix, label in tests:
    c = count_at_prefix(kb, prefix)
    print(f"  count_branches({label}) = {c}")

## 3. Adapter: Drive count_branches token by token

Simulates what happens inside vLLM: each step feeds the next token and inspects which tokens the adapter allows.

In [ ]:
adapter = VLLMXgrammarAdapter(
    tokenizer=tok,
    kb_index=kb,
    fact_pattern="Fact:",
    answer_pattern="Answer:",
    eot="\n",
    avoid_duplicates=False,
)


def drive(adapter, token_ids, label=""):
    """Feed tokens through the adapter and show allowed tokens at each step."""
    adapter.reset()
    print(f"=== {label} ===" if label else "")
    total = tok.decode(token_ids)
    print(f"Text: {total!r}")
    print(f"Tokens: {token_ids}")
    for i in range(1, len(token_ids) + 1):
        past = token_ids[:i]
        logits = torch.zeros(vocab_size)
        logits = adapter(past, logits)
        allowed = torch.where(logits != -math.inf)[0].tolist()
        tid = past[-1]
        td = tok.decode([tid])
        mode = adapter.mode
        if mode in ("branches", "branches_count"):
            allow_dec = [f"{a}({tok.decode([a])!r})" for a in allowed[:6]]
            extra = "..." if len(allowed) > 6 else ""
            print(f"  step {i:2d}: mode={mode:15s} token={tid:5d} {td!r:6s}  -> allowed: {', '.join(allow_dec)}{extra}")
        elif mode == "free":
            pass
        else:
            print(f"  step {i:2d}: mode={mode:15s} token={tid:5d} {td!r:6s}")
    print(f"  Final mode: {adapter.mode}")
    print(f"  Final text: {tok.decode(adapter.generated_tokens)}")
    print()


# Test with space-separated format (clean tokens)
ids = tok.encode("count_branches( <France> ) = 2", add_special_tokens=False)
drive(adapter, ids, "count_branches with spaces")

In [ ]:
# Test with merged-token format (no spaces after parens)
ids = tok.encode("count_branches(<France>) = 2", add_special_tokens=False)
drive(adapter, ids, "count_branches merged tokens")

In [ ]:
# Test with empty prefix
ids = tok.encode("count_branches( ) = 10", add_special_tokens=False)
drive(adapter, ids, "count_branches empty prefix")

## 4. JSON mode (Answer: trigger)

Tests the GrammarMatcher that constrains output to a JSON schema.

In [ ]:
json_schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "facts": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["answer", "facts"],
}

adapter2 = VLLMXgrammarAdapter(
    tokenizer=tok,
    kb_index=kb,
    json_schema=json_schema,
    fact_pattern="Fact:",
    answer_pattern="Answer:",
    eot="\n",
    avoid_duplicates=False,
)

# Drive through Answer: + valid JSON
ids = tok.encode('Answer: {"answer": "Paris", "facts": ["<Paris> <capital of> <France>"]}', add_special_tokens=False)
adapter2.reset()
logits = torch.zeros(vocab_size)
print("=== JSON mode ===")
print(f"Text: {tok.decode(ids)!r}")
for i in range(1, len(ids) + 1):
    past = ids[:i]
    logits = torch.zeros(vocab_size)
    logits = adapter2(past, logits)
    tid = past[-1]
    td = tok.decode([tid])
    if adapter2.mode == "json":
        allowed = torch.where(logits != -math.inf)[0].tolist()
        allow_dec = [f"{a}({tok.decode([a])!r})" for a in allowed[:5]]
        print(f"  step {i:2d}: mode=json token={tid:5d} {td!r:6s}  -> allowed: {', '.join(allow_dec)}...")
    elif adapter2.mode == "free":
        pass
    else:
        print(f"  step {i:2d}: mode={adapter2.mode:10s} token={tid:5d} {td!r:6s}")
print(f"Final mode: {adapter2.mode}")

## 5. Fact: triple mode

Demonstrates KB-constrained fact extraction with `avoid_duplicates`.

In [ ]:
adapter3 = VLLMXgrammarAdapter(
    tokenizer=tok,
    kb_index=kb,
    fact_pattern="Fact:",
    answer_pattern="Answer:",
    eot="\n",
    avoid_duplicates=True,
)

# Drive through Fact: + valid triple
ids = tok.encode('Fact: <France> <continent> <Europe>', add_special_tokens=False)
adapter3.reset()
print("=== Triple mode ===")
print(f"Text: {tok.decode(ids)!r}")
for i in range(1, len(ids) + 1):
    past = ids[:i]
    logits = torch.zeros(vocab_size)
    logits = adapter3(past, logits)
    tid = past[-1]
    td = tok.decode([tid])
    if adapter3.mode == "triple":
        allowed = torch.where(logits != -math.inf)[0].tolist()
        allow_dec = [f"{a}({tok.decode([a])!r})" for a in allowed[:5]]
        print(f"  step {i:2d}: mode=triple token={tid:5d} {td!r:6s}  -> allowed: {', '.join(allow_dec)}...")
    else:
        print(f"  step {i:2d}: mode={adapter3.mode:15s} token={tid:5d} {td!r:6s}")
print(f"Final mode: {adapter3.mode}")
print(f"Generated triples: {adapter3.generated_triples}")

## 6. Full pipeline: avoid_duplicates in action

Generates two Fact: triples — the second is a duplicate that gets blocked.

In [ ]:
adapter4 = VLLMXgrammarAdapter(
    tokenizer=tok,
    kb_index=kb,
    fact_pattern="Fact:",
    answer_pattern="Answer:",
    eot="\n",
    avoid_duplicates=True,
)

# Two Fact: lines, second is a duplicate
text = "Fact: <France> <continent> <Europe>\nFact: <France> <continent> <Europe>"
ids = tok.encode(text, add_special_tokens=False)
adapter4.reset()
print("=== Duplicate detection ===")
print(f"Text: {text!r}")
for i in range(1, len(ids) + 1):
    past = ids[:i]
    logits = torch.zeros(vocab_size)
    logits = adapter4(past, logits)
print(f"Triples generated: {len(adapter4.generated_triples)}")
for t in adapter4.generated_triples:
    print(f"  {tok.decode(t)}")
print(f"Final mode: {adapter4.mode}")
print(f"First triple recorded, duplicate blocked.")

## Summary

| Feature | Status |
|---------|--------|
| count_branches() — empty prefix | ✅ Counts all KB leaves |
| count_branches(<prefix>) | ✅ KB-constrained, close paren always allowed |
| count_branches merged tokens | ✅ Handles BPE merging of `(<` and `)>` |
| JSON mode (Answer:) | ✅ GrammarMatcher with JSON schema |
| Triple mode (Fact:) | ✅ KB-constrained with avoid_duplicates |
| avoid_duplicates | ✅ Unique triples only (dual-form tokenization) |